In [1]:
"""
Credit Agreement Definition Extractor  v3
==========================================
Extracts all defined terms and their definitions from credit agreement text.
Returns: dict  {term: str -> definition: str}

Patterns covered
----------------
 1. "Term" means / shall mean / will mean              (core list-style)
 2. "Term" shall be defined as / is defined as
 3. "Term" has the meaning [set forth / given / ascribed] [in / herein]
 4. "Term" refers to
 5. "Term" shall be construed as / to mean
 6. "Term" shall include / includes
 7. as used herein, "Term" means                       (inline)
 8. Prose (hereinafter referred to as "Term")          (inline, reversed)
 9. The term "Term" means                              (inline)
10. 'Term' means                                       (UK single-quote)

Edge cases handled
------------------
- Curly / smart quotes  \u201c\u201d  \u2018\u2019
- Legal abbreviations (N.A., LLC., Corp.) not treated as sentence ends
- Multiline definitions
- Nested quoted sub-terms inside a definition
- Parenthetical sub-clauses (a), (b), (c)
- Cross-references ("has the meaning set forth in Section X.Y")
- Semicolon-separated definition lists
- Section references (Section 8.01) not treated as terminators
- Currency amounts ($25,000,000)
- Full Article I blocks with many definitions
"""

import re



In [2]:

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _normalize(text):
    text = text.replace("\u201c", '"').replace("\u201d", '"')
    text = text.replace("\u2018", "'").replace("\u2019", "'")
    text = text.replace("\u2013", "-").replace("\u2014", "-")
    text = re.sub(r"[ \t]+", " ", text)
    return text


def _clean_term(term):
    term = term.strip().strip("\"'").strip()
    return re.sub(r"\s+", " ", term)


def _clean_definition(d):
    d = d.strip()
    # Strip trailing punctuation only when NOT part of an abbreviation
    d = re.sub(r"(?<![A-Z])[.,;]+$", "", d)
    d = re.sub(r"[ \t]+", " ", d).strip()
    return d


def _both(m):
    return m.group("term"), m.group("definition")


In [3]:

# ---------------------------------------------------------------------------
# Core regex building blocks
# ---------------------------------------------------------------------------
# A definition ends at:
#   - a semicolon
#   - end of string
#   - a newline immediately followed by the next quoted defined term

_NEXT_DEF_LINE = (
    r"\n\s*"
    r'"[^"]{1,80}"\s+'
    r"(?:shall\s+|will\s+)?"
    r"(?:mean|have\s+the\s+meaning|refer|include|be\s+defined|be\s+construed)"
)

_BODY = (
    r"(?P<definition>"
    r"(?:"
    r"(?!" + _NEXT_DEF_LINE + r")"
    r"[^;]"
    r")+?"
    r")"
)

_END = r"(?=\s*[;]|" + _NEXT_DEF_LINE + r"|\s*$)"


# ---------------------------------------------------------------------------
# Pattern library  (most specific -> most general)
# ---------------------------------------------------------------------------

PATTERNS = [

    # 1. as used herein / in this Agreement, "Term" means
    (
        r"[Aa]s\s+used\s+(?:herein|in\s+this\s+Agreement)[,\s]+"
        r'"(?P<term>[^"]{1,80})"'
        r"\s+(?:shall\s+|will\s+)?mean(?:s)?\s+"
        + _BODY + _END,
        _both,
    ),

    # 2. Prose (hereinafter referred to as "Term")
    (
        r"(?P<definition>[A-Z][^(\n]{5,120}?)"
        r"\s*\((?:each,\s*)?hereinafter\s+(?:referred\s+to\s+as|called)\s+"
        r'"(?P<term>[^"]{1,80})"\)',
        _both,
    ),

    # 3. The term "X" means
    (
        r'[Tt]he\s+term\s+"(?P<term>[^"]{1,80})"\s+'
        r"(?:shall\s+|will\s+)?mean(?:s)?\s+"
        + _BODY + _END,
        _both,
    ),

    # 4. "Term" means / shall mean / will mean  [CORE]
    (
        r'"(?P<term>[^"]{1,80})"'
        r"\s+(?:shall\s+|will\s+)?mean(?:s)?\s+"
        + _BODY + _END,
        _both,
    ),

    # 5. "Term" shall be defined as / is defined as
    (
        r'"(?P<term>[^"]{1,80})"'
        r"\s+(?:shall\s+|will\s+)?(?:be\s+)?defined\s+as\s+"
        + _BODY + _END,
        _both,
    ),

    # 6. "Term" has the meaning [verb] [in / herein]
    (
        r'"(?P<term>[^"]{1,80})"'
        r"\s+(?:shall\s+|will\s+)?(?:has|have)\s+the\s+meaning\s+"
        r"(?:(?:ascribed|assigned|given|set\s+forth|specified|provided)\s+)?"
        r"(?:to\s+it\s+)?"
        r"(?P<definition>(?:herein|in\s+[^;\n]{3,160}))"
        r"(?=\s*[;.]|\s*$|\s*\n)",
        _both,
    ),

    # 7. "Term" refers to
    (
        r'"(?P<term>[^"]{1,80})"'
        r"\s+(?:shall\s+|will\s+)?refers?\s+to\s+"
        + _BODY + _END,
        _both,
    ),

    # 8. "Term" shall be construed as / to mean
    (
        r'"(?P<term>[^"]{1,80})"'
        r"\s+shall\s+be\s+construed\s+(?:to\s+mean|as)\s+"
        + _BODY + _END,
        _both,
    ),

    # 9. "Term" shall include / includes
    (
        r'"(?P<term>[^"]{1,80})"'
        r"\s+(?:shall\s+|will\s+)?include[s]?\s+"
        + _BODY + _END,
        _both,
    ),

    # 10. UK single-quote: 'Term' means
    (
        r"'(?P<term>[^']{1,80})'"
        r"\s+(?:shall\s+|will\s+)?mean(?:s)?\s+"
        r"(?P<definition>[^;'\n]{10,}?)"
        r"(?=\s*[;.]|\s*\n|\s*$)",
        _both,
    ),
    (
        r'"(?P<term>[^"]{1,80})"\s*:\s*'
        + _BODY + _END,
        _both,
    ),
]


In [4]:

# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------

def extract_definitions(text):
    """
    Extract defined terms from credit agreement text.

    Parameters
    ----------
    text : str   Raw text of the agreement or section thereof.

    Returns
    -------
    dict         {term: definition}  First occurrence wins.
    """
    text = _normalize(text)
    definitions = {}
    for pattern, handler in PATTERNS:
        for match in re.finditer(pattern, text, re.IGNORECASE | re.DOTALL):
            term, definition = handler(match)
            term = _clean_term(term)
            definition = _clean_definition(definition)
            if term and len(term) > 1 and definition and len(definition) > 2:
                if term not in definitions:
                    definitions[term] = definition
    return definitions


In [5]:
# ===========================================================================
# 30 TEST CASES  — representative of real credit agreement drafting
# ===========================================================================

TEST_CASES = [

    # ── A: Core "means" variants ─────────────────────────────────────────

    ("01 Simple quoted term + means",
     '"Borrower" means ABC Corporation, a Delaware corporation.',
     {"Borrower"}),

    ("02 Shall mean variant",
     '"Lender" shall mean XYZ Bank, N.A., and its successors.',
     {"Lender"}),

    ("03 Will mean (future tense)",
     '"Closing Date" will mean the date on which all conditions precedent are satisfied.',
     {"Closing Date"}),

    ("04 Multi-word defined term",
     '"Change of Control" means the acquisition by any Person of more than 50% of the voting shares.',
     {"Change of Control"}),

    ("05 Hyphenated defined term",
     '"Interest-Rate" means the rate per annum applicable to any Loan.',
     {"Interest-Rate"}),

    # ── B: Alternative verb forms ─────────────────────────────────────────

    ("06 Shall be defined as",
     '"Applicable Margin" shall be defined as the percentage per annum set forth in Schedule I.',
     {"Applicable Margin"}),

    ("07 Refers to",
     '"Collateral" refers to all assets pledged pursuant to the Security Agreement.',
     {"Collateral"}),

    ("08 Shall be construed as",
     '"Indebtedness" shall be construed as any obligation for borrowed money.',
     {"Indebtedness"}),

    ("09 Shall include",
     '"Taxes" shall include all federal, state, local and foreign taxes, levies and assessments.',
     {"Taxes"}),

    # ── C: Cross-references ───────────────────────────────────────────────

    ("10 Has the meaning set forth in Section",
     '"LIBOR" shall have the meaning set forth in Section 2.05.',
     {"LIBOR"}),

    ("11 Has the meaning given in Annex",
     '"EBITDA" has the meaning given in the Financial Definitions Annex.',
     {"EBITDA"}),

    ("12 Has the meaning ascribed to it herein",
     '"Guarantor" shall have the meaning ascribed to it herein.',
     {"Guarantor"}),

    ("13 Has the meaning assigned in another agreement",
     '"Swap Agreement" has the meaning assigned in Section 1.01 of the ISDA Master Agreement.',
     {"Swap Agreement"}),

    # ── D: Inline definitions ─────────────────────────────────────────────

    ("14 As used herein (inline)",
     'As used herein, "Business Day" means any day other than Saturday, Sunday or a public holiday.',
     {"Business Day"}),

    ("15 As used in this Agreement (inline)",
     'As used in this Agreement, "Leverage Ratio" means the ratio of Total Debt to EBITDA.',
     {"Leverage Ratio"}),

    ("16 Hereinafter referred to as",
     'ABC Corporation (hereinafter referred to as "Borrower") agrees to repay the Loan.',
     {"Borrower"}),

    ("17 The term X means (narrative style)",
     'The term "Permitted Indebtedness" means indebtedness existing on the Closing Date.',
     {"Permitted Indebtedness"}),

    # ── E: Quote styles ───────────────────────────────────────────────────

    ("18 UK single-quote style",
     "'Facility' means the revolving credit facility made available under this Agreement.",
     {"Facility"}),

    ("19 Curly/smart double quotes",
     '\u201cObligations\u201d means all amounts owing to any Lender under this Agreement.',
     {"Obligations"}),

    # ── F: Definition body edge cases ─────────────────────────────────────

    ("20 Legal abbreviation N.A. inside definition",
     '"Administrative Agent" means JPMorgan Chase Bank, N.A., in its capacity as agent.',
     {"Administrative Agent"}),

    ("21 Section reference in definition (8.01)",
     '"Default" means any event specified in Section 8.01.',
     {"Default"}),

    ("22 Currency amount in definition",
     '"Threshold Amount" means $25,000,000 (or the Dollar Equivalent thereof).',
     {"Threshold Amount"}),

    ("23 Lettered sub-clauses (a)(b)(c) in definition",
     '"Permitted Acquisition" means any acquisition satisfying: (a) no Default exists, '
     '(b) the Borrower has delivered a compliance certificate, and '
     '(c) the Leverage Ratio does not exceed 3.50 to 1.00.',
     {"Permitted Acquisition"}),

    ("24 Nested quoted sub-term inside definition",
     '"Permitted Liens" means any lien described in clauses (a) through (k), '
     'including any "Statutory Lien" arising by operation of law.',
     {"Permitted Liens"}),

    ("25 Multiline definition",
     '"Material Adverse Effect" means any event, change or\n'
     'occurrence that has, or would reasonably be expected to have,\n'
     'a material adverse effect on the business of the Borrower.',
     {"Material Adverse Effect"}),

    # ── G: Multiple definitions ───────────────────────────────────────────

    ("26 Semicolon-separated list (3 terms)",
     '"Agent" means the Administrative Agent; '
     '"Commitment" means the obligation of each Lender to make Loans; '
     '"Loan" means any Term Loan or Revolving Loan.',
     {"Agent", "Commitment", "Loan"}),

    ("27 Semicolon-separated list (2 terms same line)",
     '"Sponsor" means Blackstone Inc.; "Fund" means any fund managed by Sponsor.',
     {"Sponsor", "Fund"}),

    ("Full Article I newline-separated block",
     'ARTICLE I - DEFINITIONS\n\n'
     '"Administrative Agent" means JPMorgan Chase Bank, N.A.;\n'
     '"Borrower" means Acme Corp., a Delaware corporation;\n'
     '"Credit Agreement" means this agreement, as amended or restated;\n'
     '"Facility Amount" means USD 500,000,000;\n'
     '"Maturity Date" means the date five years after the Closing Date.\n',
     {"Administrative Agent", "Borrower", "Credit Agreement", "Facility Amount", "Maturity Date"}),

    # ── H: Special patterns ───────────────────────────────────────────────

    ("29.0.1 Means each of (plural enumeration)",
     '"Lenders" means each of the financial institutions listed on Schedule 1.01.',
     {"Lenders"}),

    ("30.00.0 Long spelled-out defined term",
     'As used herein, "Consolidated Earnings Before Interest Taxes Depreciation and Amortization" '
     'means net income of the Borrower on a consolidated basis.',
     {"Consolidated Earnings Before Interest Taxes Depreciation and Amortization"}),
]



In [6]:

# ===========================================================================
# Test runner
# ===========================================================================

def run_tests():
    passed = 0
    failed = 0
    print("=" * 74)
    print("  CREDIT AGREEMENT DEFINITION EXTRACTOR v3  |  TEST SUITE")
    print("=" * 74)

    for description, text, expected in TEST_CASES:
        result = extract_definitions(text)
        found = set(result.keys())
        missing = expected - found

        ok = not missing
        passed += ok
        failed += not ok
        tag = "[PASS]" if ok else "[FAIL]"

        print(f"\nTest {description}")
        print(f"  {tag}", end="  ")
        if missing:
            print(f"Missing: {missing}")
            extra = found - expected
            if extra:
                print(f"         Extra : {extra}")
        else:
            sample = {k: (v[:65] + "..." if len(v) > 65 else v)
                      for k, v in list(result.items())[:2]}
            print(sample, "..." if len(result) > 2 else "")

    print("\n" + "=" * 74)
    bar = "#" * passed + "-" * failed
    print(f"  SCORE: {passed}/{passed+failed}  [{bar}]")
    print("=" * 74)


In [9]:


# ===========================================================================
# Demo: realistic agreement excerpt
# ===========================================================================

SAMPLE = """\
CREDIT AGREEMENT
 
dated as of January 1, 2024
 
among ACME CORPORATION (hereinafter referred to as "Borrower"),
the lenders party hereto (the "Lenders"),
and JPMORGAN CHASE BANK, N.A., as Administrative Agent.
 
ARTICLE I - DEFINITIONS
 
Section 1.01  Defined Terms.
 
As used in this Agreement, the following terms have the meanings set forth below:
 
"Administrative Agent" means JPMorgan Chase Bank, N.A., in its capacity as
administrative agent for the Lenders;
 
"Agreement" shall mean this Credit Agreement, as amended, restated, supplemented
or otherwise modified from time to time;
 
"Applicable Margin" means, with respect to any Loan, the percentage per annum
set forth in the Pricing Grid attached as Schedule 1.01;
 
"Business Day" means any day other than a Saturday, Sunday or other day on
which commercial banks in New York City are authorized or required to close;
 
"Change of Control" means the acquisition by any Person or group of Persons
of beneficial ownership of more than 50% of the outstanding voting Equity
Interests of the Borrower;
 
"Closing Date" will mean the first date on which all conditions set forth in
Section 4.01 have been satisfied or waived;
 
"Commitment" means, as to each Lender, its obligation to make Revolving Loans;
 
"Default" means any event or condition that constitutes an Event of Default or
that, upon notice or lapse of time, would become an Event of Default;
 
"EBITDA" has the meaning given in the Financial Definitions Annex attached hereto;
 
"Event of Default" has the meaning set forth in Section 8.01;
 
"Indebtedness" shall be construed as any obligation for the payment of borrowed money;
 
"Lender" means each of the financial institutions listed on Schedule 2.01;
 
"Material Adverse Effect" means any event, change or circumstance that has,
or would reasonably be expected to have, a material adverse effect on
(a) the business, operations, property or financial condition of the Borrower,
(b) the ability of the Borrower to perform its obligations under this Agreement,
or (c) the rights and remedies of the Administrative Agent or Lenders;
 
"Obligations" means all amounts owing to any Lender or the Administrative Agent
under any Loan Document;
 
"Permitted Liens" means the liens described in Schedule 7.01 hereto;
 
"Taxes" shall include all present or future taxes, levies, imposts, duties,
assessments or withholdings imposed by any Governmental Authority.

"Taxes With Dots" : shall include all present or future taxes, levies, imposts, duties,
assessments or withholdings imposed by any Governmental Authority.

"Taxes With Dots :" shall include all present or future taxes, levies, imposts, duties,
assessments or withholdings imposed by any Governmental Authority.

"SOFR" means the liens described in Schedule 7.01 hereto;

"SOFR TERM" means the liens described in Schedule 7.01 hereto;

"DAILY SIMPLE SOFR" means the liens described in Schedule 7.01 hereto;

"""

In [10]:
def demo():
    print("\n" + "=" * 74)
    print("  DEMO  |  Realistic Credit Agreement Excerpt")
    print("=" * 74)
    defs = extract_definitions(SAMPLE)
    print(f"\n  Extracted {len(defs)} definitions\n")
    for term in sorted(defs):
        defn = defs[term].replace("\n", " ")
        snippet = defn[:105] + ("..." if len(defn) > 105 else "")
        print(f'  "{term}"')
        print(f'      {snippet}\n')
    return defs


run_tests()
definations = demo()

  CREDIT AGREEMENT DEFINITION EXTRACTOR v3  |  TEST SUITE

Test 01 Simple quoted term + means
  [PASS]  {'Borrower': 'ABC Corporation, a Delaware corporation'} 

Test 02 Shall mean variant
  [PASS]  {'Lender': 'XYZ Bank, N.A., and its successors'} 

Test 03 Will mean (future tense)
  [PASS]  {'Closing Date': 'the date on which all conditions precedent are satisfied'} 

Test 04 Multi-word defined term
  [PASS]  {'Change of Control': 'the acquisition by any Person of more than 50% of the voting shar...'} 

Test 05 Hyphenated defined term
  [PASS]  {'Interest-Rate': 'the rate per annum applicable to any Loan'} 

Test 06 Shall be defined as
  [PASS]  {'Applicable Margin': 'the percentage per annum set forth in Schedule I.'} 

Test 07 Refers to
  [PASS]  {'Collateral': 'all assets pledged pursuant to the Security Agreement'} 

Test 08 Shall be construed as
  [PASS]  {'Indebtedness': 'any obligation for borrowed money'} 

Test 09 Shall include
  [PASS]  {'Taxes': 'all federal, state, local a

In [11]:
definations

{'Borrower': 'among ACME CORPORATION',
 'Administrative Agent': 'JPMorgan Chase Bank, N.A., in its capacity as\nadministrative agent for the Lenders',
 'Agreement': 'this Credit Agreement, as amended, restated, supplemented\nor otherwise modified from time to time',
 'Business Day': 'any day other than a Saturday, Sunday or other day on\nwhich commercial banks in New York City are authorized or required to close',
 'Change of Control': 'the acquisition by any Person or group of Persons\nof beneficial ownership of more than 50% of the outstanding voting Equity\nInterests of the Borrower',
 'Closing Date': 'the first date on which all conditions set forth in\nSection 4.01 have been satisfied or waived',
 'Default': 'any event or condition that constitutes an Event of Default or\nthat, upon notice or lapse of time, would become an Event of Default',
 'Lender': 'each of the financial institutions listed on Schedule 2.01',
 'Material Adverse Effect': 'any event, change or circumstance that 

In [12]:
import re
from rapidfuzz import fuzz


# ---------------------------------------------------------------------------
# NORMALIZATION
# ---------------------------------------------------------------------------

def normalize_term(t: str) -> str:
    t = t.lower().strip()
    t = re.sub(r'["\'():,.;]', '', t)
    t = re.sub(r'\s+', ' ', t)
    return t

In [13]:
def find_exact(term, definitions):
    norm = normalize_term(term)

    for k, v in definitions.items():
        if normalize_term(k) == norm:
            return [{
                "term": k,
                "definition": v,
                "score": 1.0,
                "match_type": "exact"
            }]
    return []

In [14]:
def expand_query(term):
    base = normalize_term(term)

    variants = set()

    # base variants
    variants.add(base)
    variants.add(base.title())
    variants.add(base.upper())

    # quoted
    variants.add(f'"{base}"')
    variants.add(f'"{base.title()}"')

    # colon style
    variants.add(f'{base}:')
    variants.add(f'"{base}":')

    # plural handling
    if not base.endswith('s'):
        variants.add(base + "s")

    # verb expansions
    verbs = [
        "means",
        "shall mean",
        "will mean",
        "refers to",
        "includes",
        "shall include",
        "is defined as",
        "shall be defined as",
    ]

    for v in verbs:
        variants.add(f'{base} {v}')
        variants.add(f'"{base}" {v}')

    return list(variants)

In [15]:
def fuzzy_score(a, b):
    return max(
        fuzz.ratio(a, b),
        fuzz.partial_ratio(a, b),
        fuzz.token_sort_ratio(a, b)
    )


def fuzzy_search(term, definitions, threshold=80):
    results = []
    norm_query = normalize_term(term)

    for k, v in definitions.items():
        score = fuzzy_score(norm_query, normalize_term(k))

        if score >= threshold:
            results.append({
                "term": k,
                "definition": v,
                "score": round(score / 100, 3),
                "match_type": "fuzzy"
            })

    return results

In [16]:
def search_definition(term, definitions, threshold=80):
    # 1. exact match
    exact = find_exact(term, definitions)
    if exact:
        return exact

    # 2. expanded fuzzy search
    # expanded = expand_query(term)

    results = []
    seen = set()

    matches = fuzzy_search(term, definitions, threshold)
    for m in matches:
        key = m["term"]
        if key not in seen:
            seen.add(key)
            results.append(m)

    return sorted(results, key=lambda x: -x["score"])



In [17]:
TEST_QUERIES = [

    # exact
    ("Borrower", "exact"),
    ("borrower", "exact"),
    ("BORROWER", "exact"),

    # punctuation / quotes
    ('"Borrower"', "exact"),
    ("Borrower:", "exact"),
    ('"Borrower":', "exact"),

    # plural / slight variations
    ("Borrowers", "fuzzy"),

    # spacing issues
    ("  borrower  ", "exact"),

    # multi-word
    ("Administrative Agent", "exact"),
    ("administrative agent", "exact"),

    # fuzzy partial
    ("admin agent", "fuzzy"),

    # longer terms
    ("Material Adverse Effect", "exact"),
    ("material adverse", "fuzzy"),

    # word order variation
    ("Effect Material Adverse", "fuzzy"),

    # synonym-ish noise
    ("loan facility", "fuzzy"),

    # typo
    ("borrowre", "fuzzy"),

    # colon format input
    ("Borrower:", "exact"),

    # random LLM-style query
    ("borrower shall mean", "fuzzy"),

    # another
    ("tax", "fuzzy"),

    ("SOFR", "excat"),
]


# ---------------------------------------------------------------------------
# TEST RUNNER
# ---------------------------------------------------------------------------

def run_search_tests():
    print("=" * 70)
    print("DEFINITION SEARCH TEST SUITE")
    print("=" * 70)

    for query, expected_type in TEST_QUERIES:
        results = search_definition(query, definations)

        print(f"\nQuery: {query}")

        if not results:
            print("  ❌ No match found")
            continue

        top = results[0]

        print(f"  ✅ Top Match: {top['term']}")
        print(f"     Score: {top['score']}")
        print(f"     Type : {top['match_type']}")

        if expected_type == "exact" and top["match_type"] != "exact":
            print("     ⚠️ Expected EXACT match")
        elif expected_type == "fuzzy" and top["match_type"] == "exact":
            print("     ⚠️ Expected FUZZY match")


# ---------------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    run_search_tests()

DEFINITION SEARCH TEST SUITE

Query: Borrower
  ✅ Top Match: Borrower
     Score: 1.0
     Type : exact

Query: borrower
  ✅ Top Match: Borrower
     Score: 1.0
     Type : exact

Query: BORROWER
  ✅ Top Match: Borrower
     Score: 1.0
     Type : exact

Query: "Borrower"
  ✅ Top Match: Borrower
     Score: 1.0
     Type : exact

Query: Borrower:
  ✅ Top Match: Borrower
     Score: 1.0
     Type : exact

Query: "Borrower":
  ✅ Top Match: Borrower
     Score: 1.0
     Type : exact

Query: Borrowers
  ✅ Top Match: Borrower
     Score: 1.0
     Type : fuzzy

Query:   borrower  
  ✅ Top Match: Borrower
     Score: 1.0
     Type : exact

Query: Administrative Agent
  ✅ Top Match: Administrative Agent
     Score: 1.0
     Type : exact

Query: administrative agent
  ✅ Top Match: Administrative Agent
     Score: 1.0
     Type : exact

Query: admin agent
  ❌ No match found

Query: Material Adverse Effect
  ✅ Top Match: Material Adverse Effect
     Score: 1.0
     Type : exact

Query: material a